# MMAction2 Exploratory Notebook

Adapted from OpenMMLab's official Colab tutorial for local use against an already-installed MMAction2 environment (no reinstall/bootstrap steps). In this notebook, you will:
- Perform inference with a MMAction2 recognizer.
- Train a new recognizer with a new dataset.

Let's start!

## Setup

In [1]:
from pathlib import Path

# This notebook lives in notebooks/, one level below the repository root.
ROOT = Path.cwd().parent

## Environment check

In [2]:
# Check Pytorch installation
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

# Check MMAction2 installation
import mmaction
print(mmaction.__version__)

# Check MMCV installation
from mmcv.ops import get_compiling_cuda_version, get_compiler_version
print(get_compiling_cuda_version())
print(get_compiler_version())

# Check MMEngine installation
from mmengine.utils.dl_utils import collect_env
print(collect_env())

2.13.0+cu132 True
1.2.0
13.2
GCC 13.3
OrderedDict({'sys.platform': 'linux', 'Python': '3.12.14 (main, Sep  1 2026, 14:16:52) [Clang 22.1.3 ]', 'CUDA available': True, 'MUSA available': False, 'numpy_random_seed': np.uint32(2147483648), 'GPU 0': 'NVIDIA GeForce RTX 5070', 'CUDA_HOME': None, 'GCC': 'cc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0', 'PyTorch': '2.13.0+cu132', 'PyTorch compiling details': 'PyTorch built with:\n  - GCC 13.3\n  - C++ Version: 202002\n  - Intel(R) oneAPI Math Kernel Library Version 2024.2-Product Build 20240605 for Intel(R) 64 architecture applications\n  - Intel(R) MKL-DNN v3.12.0 (Git Hash 80afa71049cd69a3df32adcccb623b12cd7baa22)\n  - OpenMP 201511 (a.k.a. OpenMP 4.5)\n  - LAPACK is enabled (usually provided by MKL)\n  - NNPACK is enabled\n  - CPU capability usage: AVX512\n  - CUDA Runtime 13.2\n  - NVCC architecture flags: -gencode;arch=compute_75,code=sm_75;-gencode;arch=compute_80,code=sm_80;-gencode;arch=compute_86,code=sm_86;-gencode;arch=compute_90,code=s

## Perform inference with a MMAction2 recognizer
MMAction2 already provides high level APIs to do inference and training.

In [3]:
import urllib.request

checkpoint_dir = ROOT / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True)

checkpoint_path = checkpoint_dir / "tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb_20220906-cd10898e.pth"
checkpoint_url = (
    "https://download.openmmlab.com/mmaction/v1.0/recognition/tsn/"
    "tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb/"
    "tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb_20220906-cd10898e.pth"
)

if not checkpoint_path.exists():
    urllib.request.urlretrieve(checkpoint_url, checkpoint_path)

In [4]:
from mmaction.apis import inference_recognizer, init_recognizer
from mmengine import Config


# Choose to use a config and initialize the recognizer
config_path = ROOT / "configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py"
config = Config.fromfile(config_path)
# Setup a checkpoint file to load
checkpoint_path = ROOT / "checkpoints/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb_20220906-cd10898e.pth"
# Initialize the recognizer
model = init_recognizer(config, str(checkpoint_path), device='cuda:0')

Loads checkpoint by local backend from path: /home/eriol/projects/mmaction2/checkpoints/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb_20220906-cd10898e.pth


In [ ]:
# Use the recognizer to do inference
from operator import itemgetter
video_path = ROOT / "videos/demo.mp4"
label_path = ROOT / "tools/data/kinetics/label_map_k400.txt"
results = inference_recognizer(model, str(video_path))

pred_scores = results.pred_score.tolist()
score_tuples = tuple(zip(range(len(pred_scores)), pred_scores))
score_sorted = sorted(score_tuples, key=itemgetter(1), reverse=True)
top5_label = score_sorted[:5]

labels = [line.strip() for line in label_path.read_text().splitlines()]
results = [(labels[k[0]], k[1]) for k in top5_label]

In [6]:
print('The top-5 labels with corresponding scores are:')
for result in results:
    print(f'{result[0]}: ', result[1])

The top-5 labels with corresponding scores are:
arm wrestling:  1.0
rock scissors paper:  1.9676210660790616e-10
shaking hands:  7.062976592475678e-11
playing chess:  1.2647963405776341e-11
stretching leg:  1.2104549133862275e-11


## Train a recognizer on customized dataset

To train a new recognizer, there are usually three things to do:
1. Support a new dataset
2. Modify the config
3. Train a new recognizer

### Support a new dataset

In this tutorial, we gives an example to convert the data into the format of existing datasets. Other methods and more advanced usages can be found in the [doc](/docs/tutorials/new_dataset.md)

Firstly, let's download a tiny dataset obtained from [Kinetics-400](https://deepmind.com/research/open-source/open-source-datasets/kinetics/). We select 30 videos with their labels as train dataset and 10 videos with their labels as test dataset.

In [ ]:
# download, decompress the data
import shutil
import urllib.request
import zipfile

data_root = ROOT / "kinetics400_tiny"
zip_path = ROOT / "kinetics400_tiny.zip"

if data_root.exists():
    shutil.rmtree(data_root)
for stale_zip in ROOT.glob("kinetics400_tiny.zip*"):
    stale_zip.unlink()

urllib.request.urlretrieve(
    "https://download.openmmlab.com/mmaction/kinetics400_tiny.zip", zip_path
)

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(ROOT)

In [ ]:
# Check the directory structure of the tiny data
def print_tree(path: Path, prefix: str = ""):
    entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, entry in enumerate(entries):
        connector = "\u2514\u2500\u2500 " if i == len(entries) - 1 else "\u251c\u2500\u2500 "
        print(f"{prefix}{connector}{entry.name}")
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "\u2502   "
            print_tree(entry, prefix + extension)

print_tree(data_root)

In [ ]:
# After downloading the data, we need to check the annotation format
print((data_root / "kinetics_tiny_train_video.txt").read_text())

According to the format defined in [`VideoDataset`](./datasets/video_dataset.py), each line indicates a sample video with the filepath and label, which are split with a whitespace.

### Modify the config

In the next step, we need to modify the config for the training.
To accelerate the process, we finetune a recognizer using a pre-trained recognizer.

In [ ]:
config_path = ROOT / "configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py"
cfg = Config.fromfile(config_path)

Given a config that trains a TSN model on kinetics400-full dataset, we need to modify some values to use it for training TSN on Kinetics400-tiny dataset.


In [ ]:
from mmengine.runner import set_random_seed

# Modify dataset type and path
cfg.data_root = str(data_root / "train")
cfg.data_root_val = str(data_root / "val")
cfg.ann_file_train = str(data_root / "kinetics_tiny_train_video.txt")
cfg.ann_file_val = str(data_root / "kinetics_tiny_val_video.txt")


cfg.test_dataloader.dataset.ann_file = str(data_root / "kinetics_tiny_val_video.txt")
cfg.test_dataloader.dataset.data_prefix.video = str(data_root / "val")

cfg.train_dataloader.dataset.ann_file = str(data_root / "kinetics_tiny_train_video.txt")
cfg.train_dataloader.dataset.data_prefix.video = str(data_root / "train")

cfg.val_dataloader.dataset.ann_file = str(data_root / "kinetics_tiny_val_video.txt")
cfg.val_dataloader.dataset.data_prefix.video = str(data_root / "val")


# Modify num classes of the model in cls_head
cfg.model.cls_head.num_classes = 2
# We can use the pre-trained TSN model
cfg.load_from = str(ROOT / "checkpoints/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb_20220906-cd10898e.pth")

# Set up working dir to save files and logs.
cfg.work_dir = str(ROOT / "tutorial_exps")

# The original learning rate (LR) is set for 8-GPU training.
# We divide it by 8 since we only use one GPU.
cfg.train_dataloader.batch_size = cfg.train_dataloader.batch_size // 16
cfg.val_dataloader.batch_size = cfg.val_dataloader.batch_size // 16
cfg.optim_wrapper.optimizer.lr = cfg.optim_wrapper.optimizer.lr / 8 / 16
cfg.train_cfg.max_epochs = 10

cfg.train_dataloader.num_workers = 2
cfg.val_dataloader.num_workers = 2
cfg.test_dataloader.num_workers = 2

# We can initialize the logger for training and have a look
# at the final config used for training
print(f'Config:\n{cfg.pretty_text}')

### Train a new recognizer

Finally, lets initialize the dataset and recognizer, then train a new recognizer!

In [ ]:
from mmengine.runner import Runner

# Create work_dir
Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)

# build the runner from config
runner = Runner.from_cfg(cfg)

# start training
runner.train()

### Understand the log
From the log, we can have a basic understanding the training process and know how well the recognizer is trained.

Firstly, the ResNet-50 backbone pre-trained on ImageNet is loaded, this is a common practice since training from scratch is more cost. The log shows that all the weights of the ResNet-50 backbone are loaded except the `fc.bias` and `fc.weight`.

Second, since the dataset we are using is small, we loaded a TSN model and finetune it for action recognition.
The original TSN is trained on original Kinetics-400 dataset which contains 400 classes but Kinetics-400 Tiny dataset only have 2 classes. Therefore, the last FC layer of the pre-trained TSN for classification has different weight shape and is not used.

Third, after training, the recognizer is evaluated by the default evaluation. The results show that the recognizer achieves 100% top1 accuracy and 100% top5 accuracy on the val dataset,
 
Not bad!

## Test the trained recognizer

After finetuning the recognizer, let's check the prediction results!

In [ ]:
runner.test()